# 04 - Physical Data Model Build and Validation

## Purpose

This notebook converts the approved audit, metric, mapping, and star-schema
specifications into reproducible physical fact and dimension tables for
Power BI.

The notebook implements:

- Source-snapshot reproducibility validation
- Approved category mappings
- LOS and financial parsing
- Data-validity flags
- Dimension-table construction
- Deterministic surrogate-key assignment
- Unknown-member key handling
- `FactDischarge` construction
- Natural-key consistency testing
- Foreign-key and orphan validation
- Fact-row reconciliation
- Physical datatype validation
- Power BI-ready Parquet exports
- Machine-readable physical-model validation outputs
- Business-readable physical-model documentation

## Inputs

Raw source:

- `data/raw/sparcs_inpatient_2023.csv`

Outputs from `01_data_audit.ipynb`:

- `outputs/data_audit/file_metadata.csv`
- `outputs/data_audit/schema.csv`
- `outputs/data_audit/numeric_audit.csv`
- `outputs/data_audit/los_distribution.csv`
- `outputs/data_audit/financial_distribution.csv`
- `outputs/data_audit/duplicate_summary.csv`
- `outputs/data_audit/facility_key_audit.csv`

Outputs from `02_metric_catalog_and_benchmark_design.ipynb`:

- `outputs/metric_catalog/category_mapping.csv`
- `outputs/metric_catalog/validation_results.csv`

Outputs from `03_star_schema_design.ipynb`:

- `outputs/star_schema/table_specification.csv`
- `outputs/star_schema/column_specification.csv`
- `outputs/star_schema/source_to_model_mapping.csv`
- `outputs/star_schema/relationship_specification.csv`
- `outputs/star_schema/key_policy.csv`
- `outputs/star_schema/schema_validation_results.csv`

## Outputs

Physical model tables:

- `outputs/physical_model/tables/FactDischarge.parquet`
- `outputs/physical_model/tables/DimHospital.parquet`
- `outputs/physical_model/tables/DimDate.parquet`
- `outputs/physical_model/tables/DimService.parquet`
- `outputs/physical_model/tables/DimCaseMix.parquet`
- `outputs/physical_model/tables/DimDiagnosis.parquet`
- `outputs/physical_model/tables/DimProcedure.parquet`
- `outputs/physical_model/tables/DimPatientSegment.parquet`
- `outputs/physical_model/tables/DimGeography.parquet`
- `outputs/physical_model/tables/DimPayer.parquet`
- `outputs/physical_model/tables/DimAdmissionContext.parquet`

Validation outputs:

- `outputs/physical_model/transformation_standards.csv`
- `outputs/physical_model/source_snapshot_validation.csv`
- `outputs/physical_model/mapping_coverage.csv`
- `outputs/physical_model/staging_profile.csv`
- `outputs/physical_model/natural_key_consistency.csv`
- `outputs/physical_model/severity_domain_validation.csv`
- `outputs/physical_model/dimension_row_counts.csv`
- `outputs/physical_model/dimension_key_validation.csv`
- `outputs/physical_model/orphan_validation.csv`
- `outputs/physical_model/unknown_key_usage.csv`
- `outputs/physical_model/data_type_validation.csv`
- `outputs/physical_model/physical_validation_results.csv`
- `outputs/physical_model/parquet_validation.csv`
- `outputs/physical_model/export_manifest.csv`
- `docs/physical_data_model.md`

## Analytical Grain

`FactDischarge` contains one row per released inpatient discharge.

No records are removed because they participate in repeated released-value
patterns. The public dataset does not contain a durable discharge identifier
that would allow those patterns to be classified as erroneous duplicates.

## Scope Boundary

This notebook physically constructs the descriptive analytical model.

It does not:

- Calculate peer-expected LOS
- Calculate peer-expected estimated cost
- Calculate final business KPIs
- Implement DAX
- Build Power BI report pages
- Train predictive models
- Create patient or longitudinal discharge identifiers
- Perform causal analysis

Peer-benchmark columns approved in Notebook 03 are created as typed null
placeholders so that Notebook 05 can populate them without changing the
physical fact-table schema.

## 1. Imports

In [1]:
from pathlib import Path
import hashlib

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 2. Project Paths

In [2]:
def find_project_root(start_path):
    """Find the repository root using the existing project charter."""
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "docs" / "project_charter.md").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Expected docs/project_charter.md "
        "in the current directory or one of its parents."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "sparcs_inpatient_2023.csv"
AUDIT_DIR = PROJECT_ROOT / "outputs" / "data_audit"
CATALOG_DIR = PROJECT_ROOT / "outputs" / "metric_catalog"
SCHEMA_DIR = PROJECT_ROOT / "outputs" / "star_schema"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "physical_model"
TABLE_DIR = OUTPUT_DIR / "tables"
WORK_DIR = OUTPUT_DIR / "_work"
DOCS_DIR = PROJECT_ROOT / "docs"
WORK_DB_PATH = WORK_DIR / "physical_build.duckdb"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"Source file not found: {DATA_PATH}"

print("Project root detected successfully.")
print("Project folder:", PROJECT_ROOT.name)
print("Data Path detected successfully.")
print("Data File:", DATA_PATH.name)
print("Audit Directory:", AUDIT_DIR.name)
print("Catalog Directory:", CATALOG_DIR.name)
print("Schema Directory:", SCHEMA_DIR.name)
print("Output Directory:", OUTPUT_DIR.name)
print("Table Directory:", TABLE_DIR.name)

Project root detected successfully.
Project folder: 04_hospital_operations_powerbi
Data Path detected successfully.
Data File: sparcs_inpatient_2023.csv
Audit Directory: data_audit
Catalog Directory: metric_catalog
Schema Directory: star_schema
Output Directory: physical_model
Table Directory: tables


### Interpretation

All paths are repository-relative.

The physical build does not depend on a user-specific Windows or OneDrive
location. Parquet files are written to a dedicated analytical-model directory
for downstream Power BI ingestion.

## 3. Load Upstream Specifications

In [3]:
input_files = {
    "file_metadata": AUDIT_DIR / "file_metadata.csv",
    "schema": AUDIT_DIR / "schema.csv",
    "numeric_audit": AUDIT_DIR / "numeric_audit.csv",
    "los_distribution": AUDIT_DIR / "los_distribution.csv",
    "financial_distribution": AUDIT_DIR / "financial_distribution.csv",
    "duplicate_summary": AUDIT_DIR / "duplicate_summary.csv",
    "facility_key_audit": AUDIT_DIR / "facility_key_audit.csv",
    "category_mapping": CATALOG_DIR / "category_mapping.csv",
    "catalog_validation": CATALOG_DIR / "validation_results.csv",
    "table_specification": SCHEMA_DIR / "table_specification.csv",
    "column_specification": SCHEMA_DIR / "column_specification.csv",
    "source_to_model_mapping": SCHEMA_DIR / "source_to_model_mapping.csv",
    "relationship_specification": SCHEMA_DIR / "relationship_specification.csv",
    "key_policy": SCHEMA_DIR / "key_policy.csv",
    "schema_validation": SCHEMA_DIR / "schema_validation_results.csv",
}

missing_input_files = [str(p) for p in input_files.values() if not p.exists()]
assert not missing_input_files, (
    "Required upstream outputs are missing:\n"
    + "\n".join(missing_input_files)
)

inputs = {name: pd.read_csv(path) for name, path in input_files.items()}
file_metadata = inputs["file_metadata"]
schema = inputs["schema"]
numeric_audit = inputs["numeric_audit"]
los_distribution = inputs["los_distribution"]
financial_distribution = inputs["financial_distribution"]
duplicate_summary = inputs["duplicate_summary"]
facility_key_audit = inputs["facility_key_audit"]
category_mapping = inputs["category_mapping"]
catalog_validation = inputs["catalog_validation"]
table_specification = inputs["table_specification"]
column_specification = inputs["column_specification"]
source_to_model_mapping = inputs["source_to_model_mapping"]
relationship_specification = inputs["relationship_specification"]
key_policy = inputs["key_policy"]
schema_validation = inputs["schema_validation"]

available_columns = set(schema["column_name"].dropna().astype(str).str.strip())
physical_table_specification = (
    table_specification.loc[table_specification["table_type"].isin(["Fact", "Dimension"])]
    .copy()
    .reset_index(drop=True)
)
physical_table_names = physical_table_specification["table_name"].tolist()

print(f"Audited source fields: {len(available_columns)}")
print(f"Approved category mappings: {len(category_mapping)}")
print(f"Physical model tables: {len(physical_table_names)}")

Audited source fields: 33
Approved category mappings: 51
Physical model tables: 11


## 4. Upstream Commit Gates

In [4]:
def normalize_boolean(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

catalog_tests_passed = normalize_boolean(catalog_validation["passed"]).fillna(False)
schema_tests_passed = normalize_boolean(schema_validation["passed"]).fillna(False)

assert catalog_tests_passed.all(), "Notebook 02 validation results contain failures."
assert schema_tests_passed.all(), "Notebook 03 schema validation results contain failures."
assert category_mapping["mapping_status"].eq("APPROVED").all(), (
    "One or more category mappings are not approved."
)
assert ~category_mapping.duplicated(["source_field", "source_value"]).any(), (
    "Category mappings are not unique by source field and source value."
)

mapped_source_fields = set(
    source_to_model_mapping["source_field"].dropna().astype(str).str.strip()
)
assert mapped_source_fields == available_columns, (
    "Notebook 03 source-to-model mapping does not exactly cover the audited source schema."
)
assert facility_key_audit.empty, (
    "Permanent Facility Id currently maps to multiple facility names. "
    "Resolve the facility-key inconsistency before physical construction."
)
assert relationship_specification["cardinality"].eq("One-to-many (1:*)").all()
assert relationship_specification["cross_filter_direction"].eq("Single").all()
assert normalize_boolean(relationship_specification["active"]).fillna(False).all()
assert relationship_specification["unknown_member_key"].astype(int).eq(0).all()

expected_physical_tables = {
    "FactDischarge", "DimHospital", "DimDate", "DimService", "DimCaseMix",
    "DimDiagnosis", "DimProcedure", "DimPatientSegment", "DimGeography",
    "DimPayer", "DimAdmissionContext",
}
assert set(physical_table_names) == expected_physical_tables, (
    "Notebook 03 physical table specification has changed."
)

print("Notebook 01, 02, and 03 commit gates passed.")

Notebook 01, 02, and 03 commit gates passed.


### Interpretation

Notebook 04 treats the approved outputs from Notebooks 01–03 as committed
design contracts.

The physical build does not silently alter metric definitions, category
mappings, table grains, relationship directions, or key policies.

## 5. Source Snapshot Reproducibility

In [5]:
def calculate_sha256(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with file_path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

assert len(file_metadata) == 1, (
    "Notebook 01 file metadata must contain exactly one source snapshot."
)
audited_sha256 = str(file_metadata.loc[0, "sha256"]).strip()
current_sha256 = calculate_sha256(DATA_PATH)

source_snapshot_validation = pd.DataFrame([
    {
        "validation_test": "Source filename matches audited snapshot",
        "passed": DATA_PATH.name == str(file_metadata.loc[0, "file_name"]).strip(),
        "details": DATA_PATH.name,
    },
    {
        "validation_test": "Source SHA-256 matches audited snapshot",
        "passed": current_sha256 == audited_sha256,
        "details": current_sha256,
    },
])

display(source_snapshot_validation)
assert source_snapshot_validation["passed"].all(), (
    "The raw source file does not match the snapshot audited in Notebook 01."
)

,validation_test,passed,details
0,Source filename matches audited snapshot,True,sparcs_inpatient_2023.csv
1,Source SHA-256 matches audited snapshot,True,d69e4b9e47fd2992c323c858cde788578b085e2ba7e0eab7ee8d961db0df97cb


## 6. Physical Transformation Standards

In [6]:
transformation_standards = pd.DataFrame([
    {
        "standard_id": "SOURCE_GRAIN_PRESERVED",
        "decision": "Every released source row is retained in FactDischarge.",
        "rationale": "The audited grain is one released inpatient discharge.",
    },
    {
        "standard_id": "NO_DUPLICATE_REMOVAL",
        "decision": "Repeated released-value patterns are not removed.",
        "rationale": (
            "No durable public discharge identifier exists to prove that repeated rows "
            "are erroneous duplicates."
        ),
    },
    {
        "standard_id": "TEXT_NORMALIZATION",
        "decision": "Text values are trimmed and blank strings become null.",
        "rationale": "This prevents whitespace from creating artificial dimension members.",
    },
    {
        "standard_id": "APPROVED_MAPPINGS_ONLY",
        "decision": "Only category mappings approved in Notebook 02 are applied.",
        "rationale": "The physical build must not introduce undocumented business grouping logic.",
    },
    {
        "standard_id": "LOS_TOP_CODE",
        "decision": (
            "Released 120+ LOS values are represented as the observable lower bound "
            "of 120 days and flagged."
        ),
        "rationale": "The exact stay duration is unavailable for top-coded rows.",
    },
    {
        "standard_id": "FINANCIAL_VALIDITY",
        "decision": "Charges and estimated costs are retained only when numeric and greater than zero.",
        "rationale": "This implements the Notebook 02 financial valid-record rules.",
    },
    {
        "standard_id": "DETERMINISTIC_KEYS",
        "decision": "Dimension surrogate keys are assigned deterministically from sorted natural-key values.",
        "rationale": "The same source snapshot should generate the same keys across clean reruns.",
    },
    {
        "standard_id": "UNKNOWN_KEY_ZERO",
        "decision": "Dimension key 0 represents Unknown / Not Available.",
        "rationale": "Missing or unresolved dimension values must not remove fact rows.",
    },
    {
        "standard_id": "NO_FACT_IDENTIFIER",
        "decision": "No durable patient or discharge identifier is invented.",
        "rationale": "The public source does not support longitudinal linkage.",
    },
    {
        "standard_id": "BENCHMARK_COLUMNS_DEFERRED",
        "decision": "Approved peer-benchmark fact columns are created as typed null placeholders.",
        "rationale": "Notebook 05 will calculate benchmark values without changing the approved fact-table schema.",
    },
    {
        "standard_id": "PARQUET_EXPORT",
        "decision": "Physical model tables are exported as compressed Parquet.",
        "rationale": "Columnar storage is appropriate for the 2.1-million-row fact table and future Power BI/Fabric workflows.",
    },
])

transformation_standards

,standard_id,decision,rationale
0,SOURCE_GRAIN_PRESERVED,Every released source row is retained in FactDischarge.,The audited grain is one released inpatient discharge.
1,NO_DUPLICATE_REMOVAL,Repeated released-value patterns are not removed.,No durable public discharge identifier exists to prove that repeated rows are erroneous duplicates.
2,TEXT_NORMALIZATION,Text values are trimmed and blank strings become null.,This prevents whitespace from creating artificial dimension members.
3,APPROVED_MAPPINGS_ONLY,Only category mappings approved in Notebook 02 are applied.,The physical build must not introduce undocumented business grouping logic.
4,LOS_TOP_CODE,Released 120+ LOS values are represented as the observable lower bound of 120 days and flagged.,The exact stay duration is unavailable for top-coded rows.
5,FINANCIAL_VALIDITY,Charges and estimated costs are retained only when numeric and greater than zero.,This implements the Notebook 02 financial valid-record rules.
6,DETERMINISTIC_KEYS,Dimension surrogate keys are assigned deterministically from sorted natural-key values.,The same source snapshot should generate the same keys across clean reruns.
7,UNKNOWN_KEY_ZERO,Dimension key 0 represents Unknown / Not Available.,Missing or unresolved dimension values must not remove fact rows.
8,NO_FACT_IDENTIFIER,No durable patient or discharge identifier is invented.,The public source does not support longitudinal linkage.
9,BENCHMARK_COLUMNS_DEFERRED,Approved peer-benchmark fact columns are created as typed null placeholders.,Notebook 05 will calculate benchmark values without changing the approved fact-table schema.


## 7. Load Raw Source with DuckDB

In [7]:
if WORK_DB_PATH.exists():
    WORK_DB_PATH.unlink()

con = duckdb.connect(database=str(WORK_DB_PATH))
raw_relation = con.read_csv(
    str(DATA_PATH),
    header=True,
    all_varchar=True,
    sample_size=100_000,
)
_ = raw_relation.create_view("sparcs_raw")

raw_schema = con.sql("DESCRIBE sparcs_raw").df()
raw_columns = set(raw_schema["column_name"].astype(str).str.strip())
source_row_count = int(
    con.sql("SELECT COUNT(*) AS row_count FROM sparcs_raw").df().loc[0, "row_count"]
)

audited_row_counts = numeric_audit["total_n"].dropna().astype(int).unique()
assert len(audited_row_counts) == 1, (
    "Notebook 01 numeric audit does not contain one consistent source-row count."
)
audited_row_count = int(audited_row_counts[0])

assert raw_columns == available_columns, "Current raw source columns do not match the audited schema."
assert source_row_count == audited_row_count, (
    f"Current source contains {source_row_count:,} rows; Notebook 01 audited {audited_row_count:,}."
)

print(f"Source rows: {source_row_count:,}")
print(f"Source columns: {len(raw_columns)}")
print("Raw source matches the audited schema and row count.")

Source rows: 2,125,754
Source columns: 33
Raw source matches the audited schema and row count.


## 8. Validate Approved Category-Mapping Coverage

In [8]:
def quote_identifier(column_name):
    return '"' + str(column_name).replace('"', '""') + '"'


def quote_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

mapping_lookup = category_mapping.loc[:, [
    "source_field", "source_value", "target_group", "mapping_action", "mapping_status"
]].copy()
con.register("category_mapping_lookup", mapping_lookup)

mapping_fields = sorted(mapping_lookup["source_field"].unique())
mapping_coverage_rows = []

for source_field in mapping_fields:
    source_sql = quote_identifier(source_field)
    field_literal = quote_literal(source_field)
    coverage = con.sql(f"""
        WITH source_values AS (
            SELECT
                COALESCE(NULLIF(TRIM({source_sql}), ''), '[MISSING]') AS source_value,
                COUNT(*) AS row_n
            FROM sparcs_raw
            GROUP BY 1
        ),
        approved_values AS (
            SELECT source_value, target_group
            FROM category_mapping_lookup
            WHERE source_field = {field_literal}
        )
        SELECT
            COUNT(*) AS observed_category_n,
            SUM(CASE WHEN approved_values.source_value IS NULL THEN 1 ELSE 0 END) AS unmapped_category_n,
            COALESCE(SUM(CASE WHEN approved_values.source_value IS NULL THEN source_values.row_n ELSE 0 END), 0) AS unmapped_row_n
        FROM source_values
        LEFT JOIN approved_values USING (source_value)
    """).df().iloc[0]

    mapping_coverage_rows.append({
        "source_field": source_field,
        "observed_category_n": int(coverage["observed_category_n"]),
        "unmapped_category_n": int(coverage["unmapped_category_n"]),
        "unmapped_row_n": int(coverage["unmapped_row_n"]),
        "passed": int(coverage["unmapped_row_n"]) == 0,
    })

mapping_coverage = pd.DataFrame(mapping_coverage_rows)
display(mapping_coverage)
assert mapping_coverage["passed"].all(), (
    "The raw source contains values that are not represented in the approved Notebook 02 category mapping."
)

,source_field,observed_category_n,unmapped_category_n,unmapped_row_n,passed
0,APR Risk of Mortality,5,0,0,True
1,APR Severity of Illness Description,5,0,0,True
2,Age Group,5,0,0,True
3,Emergency Department Indicator,2,0,0,True
4,Patient Disposition,19,0,0,True
5,Payment Typology 1,9,0,0,True
6,Type of Admission,6,0,0,True


### Interpretation

All source categories used by governed mappings must resolve to an approved
Notebook 02 target.

If a new or changed source category appears, the notebook fails rather than
silently retaining or regrouping it.

## 9. Build Clean Staging Layer

In [9]:
con.execute(r"""
CREATE OR REPLACE TABLE stg_discharge AS
WITH normalized AS (
    SELECT
        NULLIF(TRIM("Hospital Service Area"), '') AS hospital_service_area,
        NULLIF(TRIM("Hospital County"), '') AS hospital_county,
        NULLIF(TRIM("Operating Certificate Number"), '') AS operating_certificate_number,
        NULLIF(TRIM("Permanent Facility Id"), '') AS permanent_facility_id,
        NULLIF(TRIM("Facility Name"), '') AS facility_name,
        COALESCE(NULLIF(TRIM("Age Group"), ''), '[MISSING]') AS age_group_source_value,
        NULLIF(TRIM("Zip Code - 3 digits"), '') AS patient_zip3,
        NULLIF(TRIM("Gender"), '') AS gender,
        NULLIF(TRIM("Race"), '') AS race,
        NULLIF(TRIM("Ethnicity"), '') AS ethnicity,
        CASE
            WHEN REGEXP_MATCHES(TRIM("Length of Stay"), '^120\s*\+$') THEN 120.0
            ELSE TRY_CAST(NULLIF(TRIM("Length of Stay"), '') AS DOUBLE)
        END AS los_days_candidate,
        CASE
            WHEN REGEXP_MATCHES(TRIM("Length of Stay"), '^120\s*\+$') THEN 1
            ELSE 0
        END AS is_top_coded_los,
        COALESCE(NULLIF(TRIM("Type of Admission"), ''), '[MISSING]') AS admission_type_source_value,
        COALESCE(NULLIF(TRIM("Patient Disposition"), ''), '[MISSING]') AS disposition_source_value,
        TRY_CAST(NULLIF(TRIM("Discharge Year"), '') AS INTEGER) AS discharge_year,
        NULLIF(TRIM("CCSR Diagnosis Code"), '') AS ccsr_diagnosis_code,
        NULLIF(TRIM("CCSR Diagnosis Description"), '') AS ccsr_diagnosis_description,
        NULLIF(TRIM("CCSR Procedure Code"), '') AS ccsr_procedure_code,
        NULLIF(TRIM("CCSR Procedure Description"), '') AS ccsr_procedure_description,
        NULLIF(TRIM("APR DRG Code"), '') AS apr_drg_code,
        NULLIF(TRIM("APR DRG Description"), '') AS apr_drg_description,
        NULLIF(TRIM("APR MDC Code"), '') AS apr_mdc_code,
        NULLIF(TRIM("APR MDC Description"), '') AS apr_mdc_description,
        TRY_CAST(NULLIF(TRIM("APR Severity of Illness Code"), '') AS INTEGER) AS apr_severity_code_candidate,
        COALESCE(NULLIF(TRIM("APR Severity of Illness Description"), ''), '[MISSING]') AS apr_severity_source_value,
        COALESCE(NULLIF(TRIM("APR Risk of Mortality"), ''), '[MISSING]') AS apr_mortality_source_value,
        NULLIF(TRIM("APR Medical Surgical Description"), '') AS medical_surgical_classification,
        COALESCE(NULLIF(TRIM("Payment Typology 1"), ''), '[MISSING]') AS payer_source_value,
        NULLIF(TRIM("Payment Typology 2"), '') AS payment_typology_2,
        NULLIF(TRIM("Payment Typology 3"), '') AS payment_typology_3,
        NULLIF(TRIM("Birth Weight"), '') AS birth_weight_raw,
        COALESCE(NULLIF(TRIM("Emergency Department Indicator"), ''), '[MISSING]') AS ed_indicator_source_value,
        TRY_CAST(NULLIF(REPLACE(REPLACE(TRIM("Total Charges"), '$', ''), ',', ''), '') AS DECIMAL(18, 2)) AS total_charges_candidate,
        TRY_CAST(NULLIF(REPLACE(REPLACE(TRIM("Total Costs"), '$', ''), ',', ''), '') AS DECIMAL(18, 2)) AS total_costs_candidate
    FROM sparcs_raw
)
SELECT
    n.hospital_service_area,
    n.hospital_county,
    n.operating_certificate_number,
    n.permanent_facility_id,
    n.facility_name,
    age_map.target_group AS age_group,
    n.patient_zip3,
    n.gender,
    n.race,
    n.ethnicity,
    CASE
        WHEN n.los_days_candidate > 0 AND n.los_days_candidate = FLOOR(n.los_days_candidate)
        THEN CAST(n.los_days_candidate AS BIGINT)
        ELSE NULL
    END AS los_days_lower_bound,
    n.is_top_coded_los,
    CASE
        WHEN n.los_days_candidate > 0 AND n.los_days_candidate = FLOOR(n.los_days_candidate)
        THEN 1 ELSE 0
    END AS is_valid_los,
    admission_map.target_group AS admission_type_group,
    disposition_map.target_group AS disposition_group,
    n.discharge_year,
    n.ccsr_diagnosis_code,
    n.ccsr_diagnosis_description,
    n.ccsr_procedure_code,
    n.ccsr_procedure_description,
    n.apr_drg_code,
    n.apr_drg_description,
    n.apr_mdc_code,
    n.apr_mdc_description,
    CASE WHEN n.apr_severity_code_candidate BETWEEN 1 AND 4 THEN n.apr_severity_code_candidate ELSE NULL END AS apr_severity_code,
    severity_map.target_group AS apr_severity_description,
    mortality_map.target_group AS apr_mortality_risk,
    n.medical_surgical_classification,
    payer_map.target_group AS payer_group,
    n.payment_typology_2,
    n.payment_typology_3,
    n.birth_weight_raw,
    ed_map.target_group AS ed_indicator_group,
    CASE WHEN n.total_charges_candidate > 0 THEN n.total_charges_candidate ELSE NULL END AS total_charges,
    CASE WHEN n.total_charges_candidate > 0 THEN 1 ELSE 0 END AS is_valid_charge,
    CASE WHEN n.total_costs_candidate > 0 THEN n.total_costs_candidate ELSE NULL END AS total_costs,
    CASE WHEN n.total_costs_candidate > 0 THEN 1 ELSE 0 END AS is_valid_cost,
    CASE WHEN n.total_charges_candidate > 0 AND n.total_costs_candidate > 0 THEN 1 ELSE 0 END AS is_paired_financial_valid
FROM normalized AS n
LEFT JOIN category_mapping_lookup AS age_map
    ON age_map.source_field = 'Age Group' AND age_map.source_value = n.age_group_source_value
LEFT JOIN category_mapping_lookup AS admission_map
    ON admission_map.source_field = 'Type of Admission' AND admission_map.source_value = n.admission_type_source_value
LEFT JOIN category_mapping_lookup AS disposition_map
    ON disposition_map.source_field = 'Patient Disposition' AND disposition_map.source_value = n.disposition_source_value
LEFT JOIN category_mapping_lookup AS severity_map
    ON severity_map.source_field = 'APR Severity of Illness Description' AND severity_map.source_value = n.apr_severity_source_value
LEFT JOIN category_mapping_lookup AS mortality_map
    ON mortality_map.source_field = 'APR Risk of Mortality' AND mortality_map.source_value = n.apr_mortality_source_value
LEFT JOIN category_mapping_lookup AS payer_map
    ON payer_map.source_field = 'Payment Typology 1' AND payer_map.source_value = n.payer_source_value
LEFT JOIN category_mapping_lookup AS ed_map
    ON ed_map.source_field = 'Emergency Department Indicator' AND ed_map.source_value = n.ed_indicator_source_value
""")

staging_row_count = int(con.sql("SELECT COUNT(*) AS row_count FROM stg_discharge").df().loc[0, "row_count"])
assert staging_row_count == source_row_count, "Staging construction changed the source row count."
print(f"Staging rows: {staging_row_count:,}")

Staging rows: 2,125,754


## 10. Validate Staging Transformations

In [10]:
staging_profile = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(is_valid_los) AS valid_los_n,
    SUM(is_top_coded_los) AS top_coded_los_n,
    SUM(is_valid_charge) AS valid_charge_n,
    SUM(is_valid_cost) AS valid_cost_n,
    SUM(is_paired_financial_valid) AS paired_financial_valid_n,
    SUM(CASE WHEN permanent_facility_id IS NULL THEN 1 ELSE 0 END) AS missing_hospital_n,
    SUM(CASE WHEN discharge_year IS NULL THEN 1 ELSE 0 END) AS missing_year_n,
    SUM(CASE WHEN apr_drg_code IS NULL THEN 1 ELSE 0 END) AS missing_service_n,
    SUM(CASE WHEN apr_severity_code IS NULL OR apr_mortality_risk IS NULL THEN 1 ELSE 0 END) AS unresolved_case_mix_n,
    SUM(CASE WHEN ccsr_diagnosis_code IS NULL THEN 1 ELSE 0 END) AS missing_diagnosis_n,
    SUM(CASE WHEN ccsr_procedure_code IS NULL THEN 1 ELSE 0 END) AS missing_procedure_n,
    SUM(CASE WHEN patient_zip3 IS NULL THEN 1 ELSE 0 END) AS missing_geography_n
FROM stg_discharge
""").df()
display(staging_profile)

,total_rows,valid_los_n,top_coded_los_n,valid_charge_n,valid_cost_n,paired_financial_valid_n,missing_hospital_n,missing_year_n,missing_service_n,unresolved_case_mix_n,missing_diagnosis_n,missing_procedure_n,missing_geography_n
0,2125754,2125754.0,2290.0,2125754.0,2125754.0,2125754.0,5333.0,0.0,0.0,474.0,0.0,611024.0,41883.0


In [11]:
audited_los_valid_n = int(los_distribution.loc[0, "valid_n"])
audited_top_coded_n = int(los_distribution.loc[0, "top_coded_n"])
audited_charge_row = financial_distribution.loc[financial_distribution["metric"].eq("Total Charges")].iloc[0]
audited_cost_row = financial_distribution.loc[financial_distribution["metric"].eq("Total Costs")].iloc[0]
audited_charge_valid_n = int(audited_charge_row["valid_n"])
audited_cost_valid_n = int(audited_cost_row["valid_n"])
audited_charge_nonpositive_n = int(audited_charge_row["nonpositive_n"])
audited_cost_nonpositive_n = int(audited_cost_row["nonpositive_n"])

assert audited_charge_nonpositive_n == 0, (
    "Notebook 01 identified nonpositive charge values. Physical financial-validity logic requires review."
)
assert audited_cost_nonpositive_n == 0, (
    "Notebook 01 identified nonpositive estimated-cost values. Physical financial-validity logic requires review."
)

staging_summary = staging_profile.iloc[0]
assert int(staging_summary["valid_los_n"]) == audited_los_valid_n
assert int(staging_summary["top_coded_los_n"]) == audited_top_coded_n
assert int(staging_summary["valid_charge_n"]) == audited_charge_valid_n
assert int(staging_summary["valid_cost_n"]) == audited_cost_valid_n

print("LOS and financial transformations reconcile to Notebook 01.")

LOS and financial transformations reconcile to Notebook 01.


In [12]:
severity_pair_counts = con.sql("""
SELECT
    apr_severity_code,
    apr_severity_description,
    COUNT(*) AS discharge_n
FROM stg_discharge
GROUP BY
    apr_severity_code,
    apr_severity_description
ORDER BY
    apr_severity_code,
    apr_severity_description
""").df()

known_severity_groups = set(
    category_mapping.loc[
        (category_mapping["source_field"] == "APR Severity of Illness Description")
        & (category_mapping["mapping_action"] != "MISSING"),
        "target_group",
    ]
)

missing_severity_groups = set(
    category_mapping.loc[
        (category_mapping["source_field"] == "APR Severity of Illness Description")
        & (category_mapping["mapping_action"] == "MISSING"),
        "target_group",
    ]
)

severity_domain_rows = []

for _, row in severity_pair_counts.iterrows():
    code = row["apr_severity_code"]
    description = row["apr_severity_description"]

    code_is_missing = pd.isna(code)
    description_is_missing = description in missing_severity_groups

    code_valid = code_is_missing or int(code) in {1, 2, 3, 4}
    description_valid = (
        description in known_severity_groups
        or description_is_missing
    )
    missing_alignment_valid = code_is_missing == description_is_missing

    severity_domain_rows.append({
        "apr_severity_code": code,
        "apr_severity_description": description,
        "discharge_n": int(row["discharge_n"]),
        "code_valid": code_valid,
        "description_valid": description_valid,
        "missing_alignment_valid": missing_alignment_valid,
        "passed": (
            code_valid
            and description_valid
            and missing_alignment_valid
        ),
    })

severity_domain_validation = pd.DataFrame(severity_domain_rows)

display(severity_domain_validation)

assert severity_domain_validation["passed"].all(), (
    "APR severity code/description domain validation failed."
)

known_codes = set(
    severity_domain_validation.loc[
        severity_domain_validation["apr_severity_code"].notna(),
        "apr_severity_code",
    ].astype(int)
)

assert known_codes == {1, 2, 3, 4}, (
    f"Expected APR severity codes {{1, 2, 3, 4}}, found {known_codes}."
)

print("APR severity code/description domain validation passed.")

,apr_severity_code,apr_severity_description,discharge_n,code_valid,description_valid,missing_alignment_valid,passed
0,1,Minor,627605,True,True,True,True
1,2,Moderate,785246,True,True,True,True
2,3,Major,504585,True,True,True,True
3,4,Extreme,207844,True,True,True,True
4,<NA>,Missing / Not Available,474,True,True,True,True


APR severity code/description domain validation passed.


### Interpretation

The staging layer preserves the source grain and reconciles the physical LOS
and financial-validity logic to the Notebook 01 audit.

Secondary and tertiary payer values and birth weight are retained only in the
internal staging layer. They are not promoted into the approved initial
semantic model.

## 11. Validate Dimension Natural-Key Consistency

In [13]:
dimension_consistency_specs = {
    "DimHospital": {
    "keys": ["permanent_facility_id"],
    "attributes": ["facility_name", "hospital_service_area", "hospital_county"],
    },
    "DimService": {
    "keys": ["apr_drg_code","apr_mdc_code"],
    "attributes": ["apr_drg_description","apr_mdc_description","medical_surgical_classification"],
    },
    "DimCaseMix": {
        "keys": ["apr_severity_code", "apr_mortality_risk"],
        "attributes": ["apr_severity_description"],
    },
    "DimDiagnosis": {
        "keys": ["ccsr_diagnosis_code"],
        "attributes": ["ccsr_diagnosis_description"],
    },
    "DimProcedure": {
        "keys": ["ccsr_procedure_code"],
        "attributes": ["ccsr_procedure_description"],
    },
}

consistency_rows = []
for table_name, specification in dimension_consistency_specs.items():
    key_columns = specification["keys"]
    attributes = specification["attributes"]
    key_sql = ", ".join(quote_identifier(c) for c in key_columns)
    known_key_filter = " AND ".join(f"{quote_identifier(c)} IS NOT NULL" for c in key_columns)

    for attribute in attributes:
        attribute_sql = quote_identifier(attribute)
        inconsistent_n = int(
    con.sql(
        f"""
        SELECT COUNT(*) AS inconsistent_key_n
        FROM (
            SELECT {key_sql}
            FROM stg_discharge
            WHERE {known_key_filter}
            GROUP BY {key_sql}
            HAVING COUNT(
                DISTINCT CAST({attribute_sql} AS VARCHAR)
            ) > 1
        )
        """
    ).df().loc[0, "inconsistent_key_n"]
)

        consistency_rows.append({
            "table_name": table_name,
            "natural_key": "|".join(key_columns),
            "attribute": attribute,
            "inconsistent_key_n": inconsistent_n,
            "passed": inconsistent_n == 0,
        })

natural_key_consistency = pd.DataFrame(
    consistency_rows
)

display(natural_key_consistency)


assert natural_key_consistency["passed"].all(), (
    "One or more dimension natural keys map to inconsistent "
    "descriptive attributes."
)

,table_name,natural_key,attribute,inconsistent_key_n,passed
0,DimHospital,permanent_facility_id,facility_name,0,True
1,DimHospital,permanent_facility_id,hospital_service_area,0,True
2,DimHospital,permanent_facility_id,hospital_county,0,True
3,DimService,apr_drg_code|apr_mdc_code,apr_drg_description,0,True
4,DimService,apr_drg_code|apr_mdc_code,apr_mdc_description,0,True
5,DimService,apr_drg_code|apr_mdc_code,medical_surgical_classification,0,True
6,DimCaseMix,apr_severity_code|apr_mortality_risk,apr_severity_description,0,True
7,DimDiagnosis,ccsr_diagnosis_code,ccsr_diagnosis_description,0,True
8,DimProcedure,ccsr_procedure_code,ccsr_procedure_description,0,True


### Interpretation

Dimension construction does not arbitrarily select among conflicting
descriptions.

A natural-key inconsistency fails the build and requires investigation before
the dimension is materialized.

Operating Certificate Number multiplicity is retained as a diagnostic only.
It does not invalidate `DimHospital`, because Notebook 03 classifies Operating
Certificate Number as staging-only and uses Permanent Facility Id as the
hospital natural key.

## 12. Build Dimension Tables

### DimHospital

In [14]:
con.execute("""
CREATE OR REPLACE TABLE "DimHospital" AS
WITH members AS (
    SELECT
        permanent_facility_id,
        MAX(facility_name)
            AS facility_name,
        MAX(hospital_service_area) AS hospital_service_area,
        MAX(hospital_county) AS hospital_county
    FROM stg_discharge
    WHERE permanent_facility_id IS NOT NULL
    GROUP BY permanent_facility_id
)
SELECT
    CAST(0 AS BIGINT) AS hospital_key,
    CAST(NULL AS VARCHAR) AS permanent_facility_id,
    'Unknown / Not Available' AS facility_name,
    'Unknown / Not Available' AS hospital_service_area,
    'Unknown / Not Available' AS hospital_county
UNION ALL
SELECT
    ROW_NUMBER() OVER (ORDER BY permanent_facility_id) AS hospital_key,
    permanent_facility_id,
    COALESCE(facility_name, 'Unknown / Not Available'),
    COALESCE(hospital_service_area, 'Unknown / Not Available'),
    COALESCE(hospital_county, 'Unknown / Not Available')
FROM members
""")

### DimDate

In [15]:
con.execute("""
CREATE OR REPLACE TABLE "DimDate" AS
SELECT CAST(0 AS INTEGER) AS date_key, CAST(NULL AS INTEGER) AS discharge_year, 'Unknown / Not Available' AS year_label
UNION ALL
SELECT discharge_year AS date_key, discharge_year, CAST(discharge_year AS VARCHAR) AS year_label
FROM (SELECT DISTINCT discharge_year FROM stg_discharge WHERE discharge_year IS NOT NULL)
""")

### DimService

In [16]:
con.execute(
    """
    CREATE OR REPLACE TABLE "DimService" AS

    WITH members AS (
        SELECT
            apr_drg_code,
            apr_mdc_code,
            MAX(apr_drg_description)
                AS apr_drg_description,
            MAX(apr_mdc_description)
                AS apr_mdc_description,
            MAX(medical_surgical_classification)
                AS medical_surgical_classification

        FROM stg_discharge

        WHERE apr_drg_code IS NOT NULL
          AND apr_mdc_code IS NOT NULL

        GROUP BY
            apr_drg_code,
            apr_mdc_code
    )

    SELECT
        CAST(0 AS BIGINT) AS service_key,
        CAST(NULL AS VARCHAR) AS apr_drg_code,
        'Unknown / Not Available'
            AS apr_drg_description,
        CAST(NULL AS VARCHAR) AS apr_mdc_code,
        'Unknown / Not Available'
            AS apr_mdc_description,
        'Unknown / Not Available'
            AS medical_surgical_classification

    UNION ALL

    SELECT
        ROW_NUMBER() OVER (
            ORDER BY
                apr_drg_code,
                apr_mdc_code
        ) AS service_key,

        apr_drg_code,

        COALESCE(
            apr_drg_description,
            'Unknown / Not Available'
        ),

        apr_mdc_code,

        COALESCE(
            apr_mdc_description,
            'Unknown / Not Available'
        ),

        COALESCE(
            medical_surgical_classification,
            'Unknown / Not Available'
        )

    FROM members
    """
)

### DimCaseMix

In [17]:
con.execute("""
CREATE OR REPLACE TABLE "DimCaseMix" AS
WITH members AS (
    SELECT apr_severity_code, apr_mortality_risk,
           MAX(apr_severity_description) AS apr_severity_description
    FROM stg_discharge
    WHERE apr_severity_code IS NOT NULL AND apr_mortality_risk IS NOT NULL
    GROUP BY apr_severity_code, apr_mortality_risk
)
SELECT CAST(0 AS BIGINT) AS case_mix_key,
       CAST(NULL AS INTEGER) AS apr_severity_code,
       'Unknown / Not Available' AS apr_severity_description,
       'Unknown / Not Available' AS apr_mortality_risk
UNION ALL
SELECT ROW_NUMBER() OVER (ORDER BY apr_severity_code, apr_mortality_risk) AS case_mix_key,
       apr_severity_code,
       COALESCE(apr_severity_description, 'Unknown / Not Available'),
       apr_mortality_risk
FROM members
""")

### DimDiagnosis

In [18]:
con.execute("""
CREATE OR REPLACE TABLE "DimDiagnosis" AS
WITH members AS (
    SELECT ccsr_diagnosis_code, MAX(ccsr_diagnosis_description) AS ccsr_diagnosis_description
    FROM stg_discharge
    WHERE ccsr_diagnosis_code IS NOT NULL
    GROUP BY ccsr_diagnosis_code
)
SELECT CAST(0 AS BIGINT) AS diagnosis_key,
       CAST(NULL AS VARCHAR) AS ccsr_diagnosis_code,
       'Unknown / Not Available' AS ccsr_diagnosis_description
UNION ALL
SELECT ROW_NUMBER() OVER (ORDER BY ccsr_diagnosis_code) AS diagnosis_key,
       ccsr_diagnosis_code,
       COALESCE(ccsr_diagnosis_description, 'Unknown / Not Available')
FROM members
""")

### DimProcedure

In [19]:
con.execute("""
CREATE OR REPLACE TABLE "DimProcedure" AS
WITH members AS (
    SELECT ccsr_procedure_code, MAX(ccsr_procedure_description) AS ccsr_procedure_description
    FROM stg_discharge
    WHERE ccsr_procedure_code IS NOT NULL
    GROUP BY ccsr_procedure_code
)
SELECT CAST(0 AS BIGINT) AS procedure_key,
       CAST(NULL AS VARCHAR) AS ccsr_procedure_code,
       'Unknown / Not Available' AS ccsr_procedure_description
UNION ALL
SELECT ROW_NUMBER() OVER (ORDER BY ccsr_procedure_code) AS procedure_key,
       ccsr_procedure_code,
       COALESCE(ccsr_procedure_description, 'Unknown / Not Available')
FROM members
""")

### DimPatientSegment

In [20]:
con.execute("""
CREATE OR REPLACE TABLE "DimPatientSegment" AS
WITH members AS (
    SELECT DISTINCT age_group, gender, race, ethnicity
    FROM stg_discharge
    WHERE age_group IS NOT NULL AND gender IS NOT NULL AND race IS NOT NULL AND ethnicity IS NOT NULL
)
SELECT CAST(0 AS BIGINT) AS patient_segment_key,
       'Unknown / Not Available' AS age_group,
       'Unknown / Not Available' AS gender,
       'Unknown / Not Available' AS race,
       'Unknown / Not Available' AS ethnicity
UNION ALL
SELECT ROW_NUMBER() OVER (ORDER BY age_group, gender, race, ethnicity) AS patient_segment_key,
       age_group, gender, race, ethnicity
FROM members
""")

### DimGeography

In [21]:
con.execute("""
CREATE OR REPLACE TABLE "DimGeography" AS
SELECT CAST(0 AS BIGINT) AS geography_key, CAST(NULL AS VARCHAR) AS patient_zip3
UNION ALL
SELECT ROW_NUMBER() OVER (ORDER BY patient_zip3) AS geography_key, patient_zip3
FROM (SELECT DISTINCT patient_zip3 FROM stg_discharge WHERE patient_zip3 IS NOT NULL)
""")

### DimPayer

In [22]:
con.execute("""
CREATE OR REPLACE TABLE "DimPayer" AS
SELECT CAST(0 AS BIGINT) AS payer_key, 'Unknown / Not Available' AS payer_group
UNION ALL
SELECT ROW_NUMBER() OVER (ORDER BY payer_group) AS payer_key, payer_group
FROM (SELECT DISTINCT payer_group FROM stg_discharge WHERE payer_group IS NOT NULL)
""")

### DimAdmissionContext

In [23]:
con.execute("""
CREATE OR REPLACE TABLE "DimAdmissionContext" AS
WITH members AS (
    SELECT DISTINCT admission_type_group, disposition_group, ed_indicator_group
    FROM stg_discharge
    WHERE admission_type_group IS NOT NULL AND disposition_group IS NOT NULL AND ed_indicator_group IS NOT NULL
)
SELECT CAST(0 AS BIGINT) AS admission_context_key,
       'Unknown / Not Available' AS admission_type_group,
       'Unknown / Not Available' AS disposition_group,
       'Unknown / Not Available' AS ed_indicator_group
UNION ALL
SELECT ROW_NUMBER() OVER (ORDER BY admission_type_group, disposition_group, ed_indicator_group) AS admission_context_key,
       admission_type_group, disposition_group, ed_indicator_group
FROM members
""")

In [24]:
dimension_tables = (
    physical_table_specification.loc[
        physical_table_specification["table_type"].eq("Dimension"), "table_name"
    ].tolist()
)

dimension_row_counts = pd.DataFrame([
    {
        "table_name": table_name,
        "row_count": int(con.sql(
            f"SELECT COUNT(*) AS row_count FROM {quote_identifier(table_name)}"
        ).df().loc[0, "row_count"]),
    }
    for table_name in dimension_tables
])

display(dimension_row_counts)

,table_name,row_count
0,DimHospital,208
1,DimDate,2
2,DimService,483
3,DimCaseMix,17
4,DimDiagnosis,483
5,DimProcedure,321
6,DimPatientSegment,203
7,DimGeography,51
8,DimPayer,10
9,DimAdmissionContext,201


## 13. Build FactDischarge

In [25]:
con.execute("""
CREATE OR REPLACE TABLE "FactDischarge" AS
SELECT
    COALESCE(h.hospital_key, 0) AS hospital_key,
    COALESCE(d.date_key, 0) AS date_key,
    COALESCE(svc.service_key, 0) AS service_key,
    COALESCE(cm.case_mix_key, 0) AS case_mix_key,
    COALESCE(dx.diagnosis_key, 0) AS diagnosis_key,
    COALESCE(px.procedure_key, 0) AS procedure_key,
    COALESCE(ps.patient_segment_key, 0) AS patient_segment_key,
    COALESCE(geo.geography_key, 0) AS geography_key,
    COALESCE(pay.payer_key, 0) AS payer_key,
    COALESCE(ac.admission_context_key, 0) AS admission_context_key,
    s.los_days_lower_bound,
    s.is_top_coded_los,
    s.is_valid_los,
    s.total_charges,
    s.is_valid_charge,
    s.total_costs,
    s.is_valid_cost,
    s.is_paired_financial_valid,
    CAST(NULL AS DOUBLE) AS peer_expected_los_days,
    CAST(NULL AS BIGINT) AS los_peer_comparison_n,
    CAST(NULL AS DECIMAL(18, 2)) AS peer_expected_estimated_cost,
    CAST(NULL AS VARCHAR) AS los_peer_benchmark_level,
    CAST(NULL AS BIGINT) AS cost_peer_comparison_n,
    CAST(NULL AS VARCHAR) AS cost_peer_benchmark_level
FROM stg_discharge AS s
LEFT JOIN "DimHospital" AS h
    ON h.permanent_facility_id IS NOT DISTINCT FROM s.permanent_facility_id
LEFT JOIN "DimDate" AS d
    ON d.discharge_year IS NOT DISTINCT FROM s.discharge_year
LEFT JOIN "DimService" AS svc
    ON svc.apr_drg_code = s.apr_drg_code
   AND svc.apr_mdc_code = s.apr_mdc_code
LEFT JOIN "DimCaseMix" AS cm
    ON cm.apr_severity_code IS NOT DISTINCT FROM s.apr_severity_code
   AND cm.apr_mortality_risk IS NOT DISTINCT FROM s.apr_mortality_risk
LEFT JOIN "DimDiagnosis" AS dx
    ON dx.ccsr_diagnosis_code IS NOT DISTINCT FROM s.ccsr_diagnosis_code
LEFT JOIN "DimProcedure" AS px
    ON px.ccsr_procedure_code IS NOT DISTINCT FROM s.ccsr_procedure_code
LEFT JOIN "DimPatientSegment" AS ps
    ON ps.age_group IS NOT DISTINCT FROM s.age_group
   AND ps.gender IS NOT DISTINCT FROM s.gender
   AND ps.race IS NOT DISTINCT FROM s.race
   AND ps.ethnicity IS NOT DISTINCT FROM s.ethnicity
LEFT JOIN "DimGeography" AS geo
    ON geo.patient_zip3 IS NOT DISTINCT FROM s.patient_zip3
LEFT JOIN "DimPayer" AS pay
    ON pay.payer_group IS NOT DISTINCT FROM s.payer_group
LEFT JOIN "DimAdmissionContext" AS ac
    ON ac.admission_type_group IS NOT DISTINCT FROM s.admission_type_group
   AND ac.disposition_group IS NOT DISTINCT FROM s.disposition_group
   AND ac.ed_indicator_group IS NOT DISTINCT FROM s.ed_indicator_group
""")

fact_row_count = int(
    con.sql('SELECT COUNT(*) AS row_count FROM "FactDischarge"').df().loc[0, "row_count"]
)
print(f"FactDischarge rows: {fact_row_count:,}")
assert fact_row_count == source_row_count, (
    "FactDischarge does not reconcile to the raw source row count."
)

FactDischarge rows: 2,125,754


### Interpretation

`FactDischarge` preserves one physical row for every released discharge.

Missing or unresolved dimensional members resolve to key `0` rather than
causing the fact row to be removed.

Peer-benchmark columns are physically present but intentionally unpopulated.

## 14. Physical Model Validation

In [26]:
dimension_key_validation_rows = []
for _, specification in physical_table_specification.loc[
    physical_table_specification["table_type"].eq("Dimension")
].iterrows():
    table_name = specification["table_name"]
    primary_key = specification["primary_key"]
    result = con.sql(f"""
        SELECT
            COUNT(*) AS row_count,
            COUNT(DISTINCT {quote_identifier(primary_key)}) AS distinct_key_n,
            SUM(CASE WHEN {quote_identifier(primary_key)} IS NULL THEN 1 ELSE 0 END) AS null_key_n,
            SUM(CASE WHEN {quote_identifier(primary_key)} = 0 THEN 1 ELSE 0 END) AS unknown_key_n
        FROM {quote_identifier(table_name)}
    """).df().iloc[0]
    passed = (
        int(result["row_count"]) == int(result["distinct_key_n"])
        and int(result["null_key_n"]) == 0
        and int(result["unknown_key_n"]) == 1
    )
    dimension_key_validation_rows.append({
        "table_name": table_name,
        "primary_key": primary_key,
        "row_count": int(result["row_count"]),
        "distinct_key_n": int(result["distinct_key_n"]),
        "null_key_n": int(result["null_key_n"]),
        "unknown_key_n": int(result["unknown_key_n"]),
        "passed": passed,
    })

dimension_key_validation = pd.DataFrame(dimension_key_validation_rows)
display(dimension_key_validation)

,table_name,primary_key,row_count,distinct_key_n,null_key_n,unknown_key_n,passed
0,DimHospital,hospital_key,208,208,0,1,True
1,DimDate,date_key,2,2,0,1,True
2,DimService,service_key,483,483,0,1,True
3,DimCaseMix,case_mix_key,17,17,0,1,True
4,DimDiagnosis,diagnosis_key,483,483,0,1,True
5,DimProcedure,procedure_key,321,321,0,1,True
6,DimPatientSegment,patient_segment_key,203,203,0,1,True
7,DimGeography,geography_key,51,51,0,1,True
8,DimPayer,payer_key,10,10,0,1,True
9,DimAdmissionContext,admission_context_key,201,201,0,1,True


In [27]:
orphan_validation_rows = []
for _, relationship in relationship_specification.iterrows():
    dimension_table = relationship["from_table"]
    dimension_column = relationship["from_column"]
    fact_column = relationship["to_column"]
    orphan_n = int(con.sql(f"""
        SELECT COUNT(*) AS orphan_n
        FROM "FactDischarge" AS f
        LEFT JOIN {quote_identifier(dimension_table)} AS d
            ON f.{quote_identifier(fact_column)} = d.{quote_identifier(dimension_column)}
        WHERE d.{quote_identifier(dimension_column)} IS NULL
    """).df().loc[0, "orphan_n"])
    orphan_validation_rows.append({
        "relationship_name": relationship["relationship_name"],
        "dimension_table": dimension_table,
        "fact_column": fact_column,
        "orphan_row_n": orphan_n,
        "passed": orphan_n == 0,
    })

orphan_validation = pd.DataFrame(orphan_validation_rows)
display(orphan_validation)

,relationship_name,dimension_table,fact_column,orphan_row_n,passed
0,DimHospital[hospital_key] -> FactDischarge[hospital_key],DimHospital,hospital_key,0,True
1,DimDate[date_key] -> FactDischarge[date_key],DimDate,date_key,0,True
2,DimService[service_key] -> FactDischarge[service_key],DimService,service_key,0,True
3,DimCaseMix[case_mix_key] -> FactDischarge[case_mix_key],DimCaseMix,case_mix_key,0,True
4,DimDiagnosis[diagnosis_key] -> FactDischarge[diagnosis_key],DimDiagnosis,diagnosis_key,0,True
5,DimProcedure[procedure_key] -> FactDischarge[procedure_key],DimProcedure,procedure_key,0,True
6,DimPatientSegment[patient_segment_key] -> FactDischarge[patient_segment_key],DimPatientSegment,patient_segment_key,0,True
7,DimGeography[geography_key] -> FactDischarge[geography_key],DimGeography,geography_key,0,True
8,DimPayer[payer_key] -> FactDischarge[payer_key],DimPayer,payer_key,0,True
9,DimAdmissionContext[admission_context_key] -> FactDischarge[admission_context_key],DimAdmissionContext,admission_context_key,0,True


In [28]:
unknown_key_usage_rows = []
for _, relationship in relationship_specification.iterrows():
    dimension_table = relationship["from_table"]
    fact_column = relationship["to_column"]
    unknown_n = int(con.sql(f"""
        SELECT COUNT(*) AS unknown_n
        FROM "FactDischarge"
        WHERE {quote_identifier(fact_column)} = 0
    """).df().loc[0, "unknown_n"])
    unknown_key_usage_rows.append({
        "dimension_table": dimension_table,
        "fact_foreign_key": fact_column,
        "unknown_key": 0,
        "unknown_fact_row_n": unknown_n,
        "unknown_fact_row_pct": round(unknown_n / fact_row_count * 100, 4),
    })

unknown_key_usage = pd.DataFrame(unknown_key_usage_rows)
display(unknown_key_usage)

,dimension_table,fact_foreign_key,unknown_key,unknown_fact_row_n,unknown_fact_row_pct
0,DimHospital,hospital_key,0,5333,0.2509
1,DimDate,date_key,0,0,0.0000
2,DimService,service_key,0,0,0.0000
3,DimCaseMix,case_mix_key,0,474,0.0223
4,DimDiagnosis,diagnosis_key,0,0,0.0000
5,DimProcedure,procedure_key,0,611024,28.7439
6,DimPatientSegment,patient_segment_key,0,0,0.0000
7,DimGeography,geography_key,0,41883,1.9703
8,DimPayer,payer_key,0,0,0.0000
9,DimAdmissionContext,admission_context_key,0,0,0.0000


In [29]:
schema_match_rows = []
for table_name in physical_table_names:
    expected_columns = column_specification.loc[
        column_specification["table_name"].eq(table_name), "column_name"
    ].tolist()
    actual_columns = con.sql(
        f"DESCRIBE {quote_identifier(table_name)}"
    ).df()["column_name"].tolist()
    schema_match_rows.append({
        "table_name": table_name,
        "expected_column_n": len(expected_columns),
        "actual_column_n": len(actual_columns),
        "missing_columns": "|".join(sorted(set(expected_columns) - set(actual_columns))),
        "unexpected_columns": "|".join(sorted(set(actual_columns) - set(expected_columns))),
        "column_order_matches": expected_columns == actual_columns,
        "passed": expected_columns == actual_columns,
    })

physical_schema_validation = pd.DataFrame(schema_match_rows)
display(physical_schema_validation)

,table_name,expected_column_n,actual_column_n,missing_columns,unexpected_columns,column_order_matches,passed
0,FactDischarge,24,24,,,True,True
1,DimHospital,5,5,,,True,True
2,DimDate,3,3,,,True,True
3,DimService,6,6,,,True,True
4,DimCaseMix,4,4,,,True,True
5,DimDiagnosis,3,3,,,True,True
6,DimProcedure,3,3,,,True,True
7,DimPatientSegment,5,5,,,True,True
8,DimGeography,2,2,,,True,True
9,DimPayer,2,2,,,True,True


In [30]:
def data_type_is_compatible(expected_type, actual_type):
    expected_type = str(expected_type).strip()
    actual_type = str(actual_type).strip().upper()
    base_type = actual_type.split("(")[0]
    if expected_type == "Whole number":
        return base_type in {"TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT"}
    if expected_type == "Text":
        return base_type == "VARCHAR"
    if expected_type == "Fixed decimal number":
        return base_type == "DECIMAL"
    if expected_type == "Decimal number":
        return base_type in {"DOUBLE", "FLOAT", "REAL", "DECIMAL"}
    return False


data_type_validation_rows = []
for table_name in physical_table_names:
    actual_schema = con.sql(f"DESCRIBE {quote_identifier(table_name)}").df()
    actual_type_lookup = dict(zip(actual_schema["column_name"], actual_schema["column_type"]))
    expected_rows = column_specification.loc[column_specification["table_name"].eq(table_name)]
    for _, column in expected_rows.iterrows():
        column_name = column["column_name"]
        expected_type = column["data_type"]
        actual_type = actual_type_lookup.get(column_name)
        passed = actual_type is not None and data_type_is_compatible(expected_type, actual_type)
        data_type_validation_rows.append({
            "table_name": table_name,
            "column_name": column_name,
            "expected_type": expected_type,
            "actual_type": actual_type,
            "passed": passed,
        })

data_type_validation = pd.DataFrame(data_type_validation_rows)
display(data_type_validation)
assert data_type_validation["passed"].all(), (
    "One or more physical columns do not match Notebook 03 datatype requirements."
)

,table_name,column_name,expected_type,actual_type,passed
0,FactDischarge,hospital_key,Whole number,BIGINT,True
1,FactDischarge,date_key,Whole number,INTEGER,True
2,FactDischarge,service_key,Whole number,BIGINT,True
3,FactDischarge,case_mix_key,Whole number,BIGINT,True
4,FactDischarge,diagnosis_key,Whole number,BIGINT,True
...,...,...,...,...,...
56,DimPayer,payer_group,Text,VARCHAR,True
57,DimAdmissionContext,admission_context_key,Whole number,BIGINT,True
58,DimAdmissionContext,admission_type_group,Text,VARCHAR,True
59,DimAdmissionContext,disposition_group,Text,VARCHAR,True


In [31]:
fact_quality_validation = con.sql("""
SELECT
    SUM(CASE WHEN is_valid_los NOT IN (0, 1) THEN 1 ELSE 0 END) AS invalid_los_flag_n,
    SUM(CASE WHEN is_top_coded_los NOT IN (0, 1) THEN 1 ELSE 0 END) AS invalid_top_code_flag_n,
    SUM(CASE WHEN is_valid_charge NOT IN (0, 1) THEN 1 ELSE 0 END) AS invalid_charge_flag_n,
    SUM(CASE WHEN is_valid_cost NOT IN (0, 1) THEN 1 ELSE 0 END) AS invalid_cost_flag_n,
    SUM(CASE WHEN is_paired_financial_valid NOT IN (0, 1) THEN 1 ELSE 0 END) AS invalid_paired_flag_n,
    SUM(CASE WHEN (los_days_lower_bound IS NULL AND is_valid_los = 1)
                  OR (los_days_lower_bound IS NOT NULL AND is_valid_los = 0)
             THEN 1 ELSE 0 END) AS los_flag_mismatch_n,
    SUM(CASE WHEN is_top_coded_los = 1 AND (los_days_lower_bound <> 120 OR is_valid_los <> 1)
             THEN 1 ELSE 0 END) AS top_code_mismatch_n,
    SUM(CASE WHEN (total_charges IS NULL AND is_valid_charge = 1)
                  OR (total_charges IS NOT NULL AND is_valid_charge = 0)
             THEN 1 ELSE 0 END) AS charge_flag_mismatch_n,
    SUM(CASE WHEN (total_costs IS NULL AND is_valid_cost = 1)
                  OR (total_costs IS NOT NULL AND is_valid_cost = 0)
             THEN 1 ELSE 0 END) AS cost_flag_mismatch_n,
    SUM(CASE WHEN is_paired_financial_valid <>
                      CASE WHEN is_valid_charge = 1 AND is_valid_cost = 1 THEN 1 ELSE 0 END
             THEN 1 ELSE 0 END) AS paired_flag_mismatch_n,
    SUM(CASE WHEN peer_expected_los_days IS NOT NULL
                  OR los_peer_comparison_n IS NOT NULL
                  OR peer_expected_estimated_cost IS NOT NULL
                  OR los_peer_benchmark_level IS NOT NULL
                  OR cost_peer_comparison_n IS NOT NULL
                  OR cost_peer_benchmark_level IS NOT NULL
             THEN 1 ELSE 0 END) AS populated_benchmark_placeholder_n
FROM "FactDischarge"
""").df()
display(fact_quality_validation)

,invalid_los_flag_n,invalid_top_code_flag_n,invalid_charge_flag_n,invalid_cost_flag_n,invalid_paired_flag_n,los_flag_mismatch_n,top_code_mismatch_n,charge_flag_mismatch_n,cost_flag_mismatch_n,paired_flag_mismatch_n,populated_benchmark_placeholder_n
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
fact_foreign_keys = relationship_specification["to_column"].tolist()
foreign_key_null_expression = " + ".join([
    f"SUM(CASE WHEN {quote_identifier(column)} IS NULL THEN 1 ELSE 0 END)"
    for column in fact_foreign_keys
])
fact_foreign_key_null_n = int(con.sql(f"""
    SELECT {foreign_key_null_expression} AS null_foreign_key_n
    FROM "FactDischarge"
""").df().loc[0, "null_foreign_key_n"])

source_reconciliation = pd.DataFrame([
    {"layer": "Raw source", "row_count": source_row_count},
    {"layer": "Clean staging", "row_count": staging_row_count},
    {"layer": "FactDischarge", "row_count": fact_row_count},
])
display(source_reconciliation)

,layer,row_count
0,Raw source,2125754
1,Clean staging,2125754
2,FactDischarge,2125754


In [33]:
fact_quality_row = fact_quality_validation.iloc[0]
fact_quality_error_columns = [
    "invalid_los_flag_n", "invalid_top_code_flag_n", "invalid_charge_flag_n",
    "invalid_cost_flag_n", "invalid_paired_flag_n", "los_flag_mismatch_n",
    "top_code_mismatch_n", "charge_flag_mismatch_n", "cost_flag_mismatch_n",
    "paired_flag_mismatch_n",
]
fact_quality_error_n = sum(int(fact_quality_row[column]) for column in fact_quality_error_columns)

physical_validation_results = pd.DataFrame([
    {"validation_test": "Source snapshot matches Notebook 01", "passed": source_snapshot_validation["passed"].all(), "details": ""},
    {"validation_test": "Approved category mappings cover source values", "passed": mapping_coverage["passed"].all(), "details": ""},
    {"validation_test": "Staging row count reconciles to source", "passed": staging_row_count == source_row_count, "details": f"{staging_row_count} vs {source_row_count}"},
    {"validation_test": "Fact row count reconciles to source", "passed": fact_row_count == source_row_count, "details": f"{fact_row_count} vs {source_row_count}"},
    {"validation_test": "Natural-key attributes are consistent", "passed": natural_key_consistency["passed"].all(), "details": ""},
    {"validation_test": "APR severity code/description domain is valid","passed": severity_domain_validation["passed"].all(),"details": "",},
    {"validation_test": "Dimension primary keys are unique", "passed": dimension_key_validation["passed"].all(), "details": ""},
    {"validation_test": "Each dimension contains exactly one key 0 member", "passed": dimension_key_validation["unknown_key_n"].eq(1).all(), "details": ""},
    {"validation_test": "Fact foreign keys contain no null values", "passed": fact_foreign_key_null_n == 0, "details": str(fact_foreign_key_null_n)},
    {"validation_test": "Fact foreign keys contain no orphan values", "passed": orphan_validation["passed"].all(), "details": ""},
    {"validation_test": "Physical columns match Notebook 03", "passed": physical_schema_validation["passed"].all(), "details": ""},
    {"validation_test": "Physical datatypes match Notebook 03", "passed": data_type_validation["passed"].all(), "details": ""},
    {"validation_test": "LOS and financial flags are internally consistent", "passed": fact_quality_error_n == 0, "details": str(fact_quality_error_n)},
    {"validation_test": "Peer benchmark columns remain unpopulated", "passed": int(fact_quality_row["populated_benchmark_placeholder_n"]) == 0, "details": str(int(fact_quality_row["populated_benchmark_placeholder_n"]))},
])

display(physical_validation_results)
assert physical_validation_results["passed"].all(), (
    "The physical data model failed one or more validation tests."
)
print("All physical-model validation gates passed.")

,validation_test,passed,details
0,Source snapshot matches Notebook 01,True,
1,Approved category mappings cover source values,True,
2,Staging row count reconciles to source,True,2125754 vs 2125754
3,Fact row count reconciles to source,True,2125754 vs 2125754
4,Natural-key attributes are consistent,True,
5,APR severity code/description domain is valid,True,
6,Dimension primary keys are unique,True,
7,Each dimension contains exactly one key 0 member,True,
8,Fact foreign keys contain no null values,True,0
9,Fact foreign keys contain no orphan values,True,


All physical-model validation gates passed.


## 15. Export Power BI-Ready Tables

In [34]:
def sql_path_literal(path):
    return "'" + Path(path).resolve().as_posix().replace("'", "''") + "'"


table_export_rows = []
for table_name in physical_table_names:
    output_path = TABLE_DIR / f"{table_name}.parquet"
    if output_path.exists():
        output_path.unlink()
    con.execute(f"""
        COPY {quote_identifier(table_name)}
        TO {sql_path_literal(output_path)}
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    assert output_path.exists() and output_path.stat().st_size > 0
    row_count = int(con.sql(
        f"SELECT COUNT(*) AS row_count FROM {quote_identifier(table_name)}"
    ).df().loc[0, "row_count"])
    table_export_rows.append({
        "file_name": output_path.name,
        "artifact_type": "Physical model table",
        "table_name": table_name,
        "row_count": row_count,
        "file_size_mb": round(output_path.stat().st_size / (1024 ** 2), 2),
        "output_path": output_path.relative_to(PROJECT_ROOT).as_posix(),
    })

table_export_manifest = pd.DataFrame(table_export_rows)
display(table_export_manifest)

,file_name,artifact_type,table_name,row_count,file_size_mb,output_path
0,FactDischarge.parquet,Physical model table,FactDischarge,2125754,28.02,outputs/physical_model/tables/FactDischarge.parquet
1,DimHospital.parquet,Physical model table,DimHospital,208,0.01,outputs/physical_model/tables/DimHospital.parquet
2,DimDate.parquet,Physical model table,DimDate,2,0.00,outputs/physical_model/tables/DimDate.parquet
3,DimService.parquet,Physical model table,DimService,483,0.01,outputs/physical_model/tables/DimService.parquet
4,DimCaseMix.parquet,Physical model table,DimCaseMix,17,0.00,outputs/physical_model/tables/DimCaseMix.parquet
5,DimDiagnosis.parquet,Physical model table,DimDiagnosis,483,0.01,outputs/physical_model/tables/DimDiagnosis.parquet
6,DimProcedure.parquet,Physical model table,DimProcedure,321,0.01,outputs/physical_model/tables/DimProcedure.parquet
7,DimPatientSegment.parquet,Physical model table,DimPatientSegment,203,0.00,outputs/physical_model/tables/DimPatientSegment.parquet
8,DimGeography.parquet,Physical model table,DimGeography,51,0.00,outputs/physical_model/tables/DimGeography.parquet
9,DimPayer.parquet,Physical model table,DimPayer,10,0.00,outputs/physical_model/tables/DimPayer.parquet


In [35]:
validation_exports = {
    "transformation_standards.csv": transformation_standards,
    "source_snapshot_validation.csv": source_snapshot_validation,
    "mapping_coverage.csv": mapping_coverage,
    "staging_profile.csv": staging_profile,
    "natural_key_consistency.csv": natural_key_consistency,
    "severity_domain_validation.csv": severity_domain_validation,
    "dimension_row_counts.csv": dimension_row_counts,
    "dimension_key_validation.csv": dimension_key_validation,
    "orphan_validation.csv": orphan_validation,
    "unknown_key_usage.csv": unknown_key_usage,
    "data_type_validation.csv": data_type_validation,
    "physical_validation_results.csv": physical_validation_results,
}

validation_export_rows = []
for file_name, dataframe in validation_exports.items():
    output_path = OUTPUT_DIR / file_name
    dataframe.to_csv(output_path, index=False)
    assert output_path.exists() and output_path.stat().st_size > 0
    validation_export_rows.append({
        "file_name": file_name,
        "artifact_type": "Validation output",
        "table_name": "",
        "row_count": len(dataframe),
        "file_size_mb": round(output_path.stat().st_size / (1024 ** 2), 4),
        "output_path": output_path.relative_to(PROJECT_ROOT).as_posix(),
    })

validation_export_manifest = pd.DataFrame(validation_export_rows)

## 16. Validate Parquet Exports

In [36]:
parquet_validation_rows = []
for _, export in table_export_manifest.iterrows():
    table_name = export["table_name"]
    parquet_path = PROJECT_ROOT / export["output_path"]
    parquet_row_count = int(con.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM read_parquet({sql_path_literal(parquet_path)})
    """).df().loc[0, "row_count"])
    parquet_columns = con.sql(f"""
        DESCRIBE SELECT * FROM read_parquet({sql_path_literal(parquet_path)})
    """).df()["column_name"].tolist()
    expected_columns = column_specification.loc[
        column_specification["table_name"].eq(table_name), "column_name"
    ].tolist()
    internal_row_count = int(export["row_count"])
    parquet_validation_rows.append({
        "table_name": table_name,
        "internal_row_count": internal_row_count,
        "parquet_row_count": parquet_row_count,
        "row_count_matches": internal_row_count == parquet_row_count,
        "column_schema_matches": expected_columns == parquet_columns,
        "passed": internal_row_count == parquet_row_count and expected_columns == parquet_columns,
    })

parquet_validation = pd.DataFrame(parquet_validation_rows)
display(parquet_validation)
assert parquet_validation["passed"].all(), (
    "One or more exported Parquet tables failed reconciliation."
)

PARQUET_VALIDATION_PATH = OUTPUT_DIR / "parquet_validation.csv"
parquet_validation.to_csv(PARQUET_VALIDATION_PATH, index=False)
assert PARQUET_VALIDATION_PATH.exists()

,table_name,internal_row_count,parquet_row_count,row_count_matches,column_schema_matches,passed
0,FactDischarge,2125754,2125754,True,True,True
1,DimHospital,208,208,True,True,True
2,DimDate,2,2,True,True,True
3,DimService,483,483,True,True,True
4,DimCaseMix,17,17,True,True,True
5,DimDiagnosis,483,483,True,True,True
6,DimProcedure,321,321,True,True,True
7,DimPatientSegment,203,203,True,True,True
8,DimGeography,51,51,True,True,True
9,DimPayer,10,10,True,True,True


In [37]:
parquet_validation_manifest_row = pd.DataFrame([{
    "file_name": PARQUET_VALIDATION_PATH.name,
    "artifact_type": "Validation output",
    "table_name": "",
    "row_count": len(parquet_validation),
    "file_size_mb": round(PARQUET_VALIDATION_PATH.stat().st_size / (1024 ** 2), 4),
    "output_path": PARQUET_VALIDATION_PATH.relative_to(PROJECT_ROOT).as_posix(),
}])

export_manifest = pd.concat(
    [table_export_manifest, validation_export_manifest, parquet_validation_manifest_row],
    ignore_index=True,
)
EXPORT_MANIFEST_PATH = OUTPUT_DIR / "export_manifest.csv"
export_manifest.to_csv(EXPORT_MANIFEST_PATH, index=False)
display(export_manifest)

,file_name,artifact_type,table_name,row_count,file_size_mb,output_path
0,FactDischarge.parquet,Physical model table,FactDischarge,2125754,28.0200,outputs/physical_model/tables/FactDischarge.parquet
1,DimHospital.parquet,Physical model table,DimHospital,208,0.0100,outputs/physical_model/tables/DimHospital.parquet
2,DimDate.parquet,Physical model table,DimDate,2,0.0000,outputs/physical_model/tables/DimDate.parquet
3,DimService.parquet,Physical model table,DimService,483,0.0100,outputs/physical_model/tables/DimService.parquet
4,DimCaseMix.parquet,Physical model table,DimCaseMix,17,0.0000,outputs/physical_model/tables/DimCaseMix.parquet
5,DimDiagnosis.parquet,Physical model table,DimDiagnosis,483,0.0100,outputs/physical_model/tables/DimDiagnosis.parquet
6,DimProcedure.parquet,Physical model table,DimProcedure,321,0.0100,outputs/physical_model/tables/DimProcedure.parquet
7,DimPatientSegment.parquet,Physical model table,DimPatientSegment,203,0.0000,outputs/physical_model/tables/DimPatientSegment.parquet
8,DimGeography.parquet,Physical model table,DimGeography,51,0.0000,outputs/physical_model/tables/DimGeography.parquet
9,DimPayer.parquet,Physical model table,DimPayer,10,0.0000,outputs/physical_model/tables/DimPayer.parquet


## 17. Generate `docs/physical_data_model.md`

In [38]:
def dataframe_to_markdown(dataframe, columns):
    selected = dataframe.loc[:, columns].fillna("").astype(str)

    def escape_value(value):
        return value.replace("|", r"\|").replace("\n", " ")

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"

    rows = [
        "| " + " | ".join(
            escape_value(value)
            for value in row
        ) + " |"
        for row in selected.itertuples(
            index=False,
            name=None
        )
    ]

    return "\n".join(
        [header, separator, *rows]
    )


transformation_markdown = dataframe_to_markdown(
    transformation_standards,
    ["standard_id", "decision", "rationale"],
)

dimension_count_markdown = dataframe_to_markdown(
    dimension_row_counts,
    ["table_name", "row_count"],
)

unknown_usage_markdown = dataframe_to_markdown(
    unknown_key_usage,
    [
        "dimension_table",
        "fact_foreign_key",
        "unknown_fact_row_n",
        "unknown_fact_row_pct",
    ],
)

validation_markdown = dataframe_to_markdown(
    physical_validation_results,
    ["validation_test", "passed", "details"],
)

parquet_markdown = dataframe_to_markdown(
    table_export_manifest,
    [
        "table_name",
        "row_count",
        "file_size_mb",
        "output_path",
    ],
)


physical_model_document = "\n".join([
    "# Physical Data Model — Hospital Operations & Cost Efficiency",
    "",
    "## Purpose",
    "",
    (
        "This document describes the reproducible physical "
        "implementation of the approved SPARCS analytical star schema."
    ),
    "",
    "## Analytical Grain",
    "",
    "`FactDischarge` contains one row per released inpatient discharge.",
    "",
    f"- Raw source rows: {source_row_count:,}",
    f"- Clean staging rows: {staging_row_count:,}",
    f"- FactDischarge rows: {fact_row_count:,}",
    "",
    "## Transformation Standards",
    "",
    transformation_markdown,
    "",
    "## Dimension Row Counts",
    "",
    dimension_count_markdown,
    "",
    "## Unknown-Member Usage",
    "",
    unknown_usage_markdown,
    "",
    "## Physical Validation Results",
    "",
    validation_markdown,
    "",
    "## Power BI-Ready Parquet Tables",
    "",
    parquet_markdown,
    "",
    "## Deferred Benchmark Columns",
    "",
    (
        "`FactDischarge` contains the approved peer-benchmark "
        "columns as typed null placeholders."
    ),
    "",
    (
        "Notebook 05 will calculate leave-one-facility-out LOS "
        "and estimated-cost peer expectations."
    ),
])


PHYSICAL_MODEL_DOCUMENT_PATH = (
    DOCS_DIR / "physical_data_model.md"
)

PHYSICAL_MODEL_DOCUMENT_PATH.write_text(
    physical_model_document,
    encoding="utf-8",
)

assert (
    PHYSICAL_MODEL_DOCUMENT_PATH.exists()
    and PHYSICAL_MODEL_DOCUMENT_PATH.stat().st_size > 0
)

print(
    "Physical-model documentation created:",
    PHYSICAL_MODEL_DOCUMENT_PATH.relative_to(PROJECT_ROOT),
)

Physical-model documentation created: docs\physical_data_model.md


In [39]:
document_manifest_row = pd.DataFrame([{
    "file_name": PHYSICAL_MODEL_DOCUMENT_PATH.name,
    "artifact_type": "Documentation",
    "table_name": "",
    "row_count": pd.NA,
    "file_size_mb": round(PHYSICAL_MODEL_DOCUMENT_PATH.stat().st_size / (1024 ** 2), 4),
    "output_path": PHYSICAL_MODEL_DOCUMENT_PATH.relative_to(PROJECT_ROOT).as_posix(),
}])

final_export_manifest = pd.concat([export_manifest, document_manifest_row], ignore_index=True)
final_export_manifest.to_csv(EXPORT_MANIFEST_PATH, index=False)
for relative_path in final_export_manifest["output_path"]:
    assert (PROJECT_ROOT / relative_path).exists()

display(final_export_manifest)

,file_name,artifact_type,table_name,row_count,file_size_mb,output_path
0,FactDischarge.parquet,Physical model table,FactDischarge,2125754,28.0200,outputs/physical_model/tables/FactDischarge.parquet
1,DimHospital.parquet,Physical model table,DimHospital,208,0.0100,outputs/physical_model/tables/DimHospital.parquet
2,DimDate.parquet,Physical model table,DimDate,2,0.0000,outputs/physical_model/tables/DimDate.parquet
3,DimService.parquet,Physical model table,DimService,483,0.0100,outputs/physical_model/tables/DimService.parquet
4,DimCaseMix.parquet,Physical model table,DimCaseMix,17,0.0000,outputs/physical_model/tables/DimCaseMix.parquet
5,DimDiagnosis.parquet,Physical model table,DimDiagnosis,483,0.0100,outputs/physical_model/tables/DimDiagnosis.parquet
6,DimProcedure.parquet,Physical model table,DimProcedure,321,0.0100,outputs/physical_model/tables/DimProcedure.parquet
7,DimPatientSegment.parquet,Physical model table,DimPatientSegment,203,0.0000,outputs/physical_model/tables/DimPatientSegment.parquet
8,DimGeography.parquet,Physical model table,DimGeography,51,0.0000,outputs/physical_model/tables/DimGeography.parquet
9,DimPayer.parquet,Physical model table,DimPayer,10,0.0000,outputs/physical_model/tables/DimPayer.parquet


## 18. Clean Temporary Build Artifacts

In [40]:
con.close()
if WORK_DB_PATH.exists():
    WORK_DB_PATH.unlink()
try:
    WORK_DIR.rmdir()
except OSError:
    pass

print("Temporary DuckDB build artifacts removed.")
print("Physical model outputs retained successfully.")

Temporary DuckDB build artifacts removed.
Physical model outputs retained successfully.


# Final Notebook Summary

## Work Completed

- Loaded and validated committed outputs from Notebooks 01–03.
- Confirmed that the raw source file matches the audited SHA-256 snapshot.
- Confirmed that the current source schema and row count match Notebook 01.
- Applied only approved Notebook 02 category mappings.
- Validated complete mapping coverage before transformation.
- Standardized text and missing values.
- Parsed LOS and preserved the `120 +` observable lower bound.
- Created LOS top-code and validity flags.
- Parsed positive charges and estimated costs.
- Created charge, cost, and paired-financial validity flags.
- Retained repeated released-value records rather than deleting apparent duplicates.
- Validated natural-key-to-description consistency before dimension construction.
- Constructed all 10 approved physical dimensions.
- Assigned deterministic whole-number surrogate keys.
- Reserved dimension key `0` for Unknown / Not Available members.
- Constructed `FactDischarge` at one row per released inpatient discharge.
- Confirmed complete fact-row reconciliation to the raw source.
- Validated dimension-key uniqueness.
- Validated foreign-key completeness and absence of orphan keys.
- Validated physical columns and datatypes against Notebook 03.
- Created approved peer-benchmark columns as typed null placeholders.
- Exported Power BI-ready Parquet tables.
- Re-read and reconciled exported Parquet tables.
- Generated `docs/physical_data_model.md`.

## Remaining Deferred Work

Notebook 05 must calculate:

- `peer_expected_los_days`
- `los_peer_comparison_n`
- `los_peer_benchmark_level`
- `peer_expected_estimated_cost`
- `cost_peer_comparison_n`
- `cost_peer_benchmark_level`

using the approved leave-one-facility-out peer methodology.

The following also remain downstream:

- Explicit DAX measures
- Small-cell suppression implementation in Power BI
- Power BI relationship verification
- Semantic-model memory testing
- Dashboard development
- Predictive expected-LOS modeling
- Versioned model scoring
- Multi-year compatibility testing
- Microsoft Fabric implementation

## Scope Boundary

This notebook constructs and validates the physical descriptive star schema.

It does not calculate hospital performance, peer rankings, final KPIs,
predictive-model expectations, or causal effects.